In [1]:
import asyncio
from collections import defaultdict
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [2]:
import sys
from pathlib import Path

# Asumiendo que notebooks/ está dentro de la raíz del proyecto
root_path = Path().resolve().parent  # sube un nivel
sys.path.append(str(root_path))

In [3]:
from agents.classifier_agent import ClassifierAgent
from agents.aggregator_agent import AggregatorAgent
from data.dataset_registry import DatasetRegistry
from data.loaders.sklearn_loader import SklearnLoader

In [ ]:
import asyncio
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

from data.dataset_registry import DatasetRegistry
from data.loaders.sklearn_loader import SklearnLoader
from agents.classifier_agent import ClassifierAgent
from agents.aggregator_agent import AggregatorAgent
from explainers.shap_explainer import ShapExplainer
from models.sklearn_model import SklearnModel

# Visualización
from visualization.results import plot_metrics_over_time
from visualization.explanations import plot_explanation_similarity, plot_explanation_divergence

async def run_test():
    # ------------------- Dataset -------------------
    registry = DatasetRegistry()
    dataset_id = "iris"
    registry.register(dataset_id, SklearnLoader(load_iris))
    X, y, meta = registry.load(dataset_id)
    print(f"[Dataset] Cargado {dataset_id}: {X.shape}")

    instance = X[:1]

    # ------------------- Clasificadores -------------------
    classifier_ids = ["clf1", "clf2", "clf3"]
    classifiers = {
        cid: ClassifierAgent(
            agent_id=cid,
            model=SklearnModel(RandomForestClassifier(n_estimators=10, random_state=42)),
            explainers=[ShapExplainer()],
            dataset_id=dataset_id,
            registry=registry
        )
        for cid in classifier_ids
    }

    # ------------------- Agregador -------------------
    aggregator = AggregatorAgent(
        classifier_ids=classifier_ids,
        max_iterations=2  # <- ahora 2 iteraciones
    )

    # ------------------- Queues -------------------
    queues = {cid: agent.inbox for cid, agent in classifiers.items()}
    queues["aggregator"] = aggregator.inbox

    # ------------------- Setup -------------------
    await asyncio.gather(*(agent.setup() for agent in classifiers.values()))

    # ------------------- Tasks -------------------
    tasks = [asyncio.create_task(agent.run(queues)) for agent in classifiers.values()]
    tasks.append(asyncio.create_task(aggregator.run(queues, instance)))

    await asyncio.gather(*tasks)

    print("\n[Test] Finalizado ✅\n")

    # ------------------- Visualizaciones -------------------
    # Métricas por agente
    for agent in classifiers.values():
        plot_metrics_over_time(agent.metrics_history, metric_name="accuracy")
        plot_explanation_similarity(agent)

    # Divergencia global de explicaciones
    plot_explanation_divergence(classifiers)


In [ ]:
await run_test()


[Dataset] Cargado iris: (150, 4)
[clf1] Setup iniciado
[clf1] Entrenado. Accuracy inicial=0.967
[clf2] Setup iniciado
[clf2] Entrenado. Accuracy inicial=0.867
[clf3] Setup iniciado
[clf3] Entrenado. Accuracy inicial=0.900
[Aggregator] Inicio

[Aggregator] Iteración 0
  ↳ {'agent_id': 'clf1', 'iteration': 0, 'prediction': 0, 'explanations': [{'explainer': 'shap', 'scope': 'local', 'instance_id': 0, 'prediction': [0], 'confidence': [1.0], 'details': {'type': 'feature_importance', 'feature_names': None, 'values': [-0.06483333333333326, -0.025000000000000064, -0.09383333333333337, -0.1443333333333333]}}], 'metrics': {'iteration': 0, 'accuracy': 0.9666666666666667}}
  ↳ {'agent_id': 'clf2', 'iteration': 0, 'prediction': 0, 'explanations': [{'explainer': 'shap', 'scope': 'local', 'instance_id': 0, 'prediction': [0], 'confidence': [1.0], 'details': {'type': 'feature_importance', 'feature_names': None, 'values': [-0.014833333333333143, -0.013666666666666678, -0.14333333333333337, -0.1611666666